# Skin Cancer Prediction 🏻

## 1. Datasets & DataLoaders

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder

#DataSet
training_data = ImageFolder(root='data/train', transform = transforms.ToTensor())
test_data = ImageFolder(root = 'data/test', transform = transforms.ToTensor())

# DataLoaders
train_dataloader = DataLoader(training_data, batch_size = 40, shuffle = True)
test_dataloader = DataLoader(test_data, batch_size = 40, shuffle = False)

## 2. Build Model

In [2]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=11, stride=4, padding = 1),
            nn.MaxPool2d(5, 5),
            nn.Conv2d(32, 64, kernel_size=5, stride=1),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(576, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 10),
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NeuralNetwork().to(device)
print(device)

cuda


## 3. Optimization

In [3]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


## 4. Main

In [4]:
learning_rate = 0.005
batch_size = 40
epochs = 20

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.305930  [   40/ 2637]
Test Error: 
 Accuracy: 56.0%, Avg loss: 0.684455 

Epoch 2
-------------------------------
loss: 0.712063  [   40/ 2637]
Test Error: 
 Accuracy: 54.5%, Avg loss: 0.692513 

Epoch 3
-------------------------------
loss: 0.680637  [   40/ 2637]
Test Error: 
 Accuracy: 45.5%, Avg loss: 0.699905 

Epoch 4
-------------------------------
loss: 0.681012  [   40/ 2637]
Test Error: 
 Accuracy: 54.8%, Avg loss: 0.687685 

Epoch 5
-------------------------------
loss: 0.676291  [   40/ 2637]
Test Error: 
 Accuracy: 54.6%, Avg loss: 0.687422 

Epoch 6
-------------------------------
loss: 0.727712  [   40/ 2637]
Test Error: 
 Accuracy: 59.3%, Avg loss: 0.688415 

Epoch 7
-------------------------------
loss: 0.687651  [   40/ 2637]
Test Error: 
 Accuracy: 55.5%, Avg loss: 0.680380 

Epoch 8
-------------------------------
loss: 0.680533  [   40/ 2637]
Test Error: 
 Accuracy: 56.7%, Avg loss: 0.666365 

Epoch 9
----------------

## 5. Save and load

In [ ]:
import torchvision.models as models
model = models.vgg16(weights='IMAGENET1K_V1')
torch.save(model.state_dict(), 'model_weights_cancer.pth')

In [ ]:
model = torch.load('model_weights_cancer.pth', weights_only=False)
print("Model loaded")